# Bridges and Cut Vertices in Graph Theory

In graph theory, **bridges** and **cut vertices** are critical concepts for understanding the structure and resilience of graphs.

- **Bridges**: A bridge, or a cut-edge, is an edge in a graph whose removal increases the number of connected components. Bridges are crucial for identifying weak points in a network, as their failure can disconnect parts of the graph.

- **Cut Vertices**: Cut vertices (also called articulation points) are vertices whose removal disconnects the graph into two or more components. Vertex cuts provide insight into the graph's connectivity and robustness.

**Note**: This notebook uses topologic_fast's native `Graph.Bridges` and `Graph.CutVertices` implementations using Tarjan's algorithm.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict, deque

## Create a Graph from an Adjacency Matrix

We'll use the same high school building layout as in the Betweenness Centrality example.

In [ ]:
# Adjacency Matrix for a fictional High School
adjacencyMatrix = [[0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [1,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0],
                   [0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0],
                   [0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0],
                   [1,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [1,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0],
                   [0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0],
                   [0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
                   [0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0],
                   [0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1],
                   [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0]]

node_names = [
    "Daycare_Room",
    "Elementary_School",
    "High_School_Wing",
    "Library",
    "Corridor_1",
    "Study_Room",
    "Special_Education_Room",
    "Swimming_Pool",
    "Gymnasium",
    "Outdoor_Space",
    "Sports_Field",
    "Corridor_2",
    "Main_Entrance_Hall",
    "Lobby",
    "Administrative_Office",
    "Principals_Office",
    "Counseling_Office",
    "Corridor_3",
    "Cafeteria",
    "Corridor_4",
    "Kitchen",
    "Corridor_5",
    "Workshop",
    "Corridor_6"]

print(f"Number of nodes: {len(node_names)}")
print(f"Adjacency matrix dimensions: {len(adjacencyMatrix)}x{len(adjacencyMatrix[0])}")

## Build the Graph

We'll use `Graph.ByAdjacencyMatrix` to create the graph directly from the adjacency matrix, and `Graph.Reshape` for the visualization layout.

In [ ]:
# Create the graph directly from the adjacency matrix using topologic_fast
graph = tf.Graph.ByAdjacencyMatrix(adjacencyMatrix)

print(f"Graph created with {graph.Order()} vertices and {graph.Size()} edges")
print(f"Graph density: {graph.Density():.3f}")
print(f"Graph diameter: {graph.Diameter()}")

# Use Graph.Reshape to get spring layout positions for visualization
positions_dict = graph.Reshape(layout_type="spring2d", iterations=100)

# Convert to numpy array for visualization
positions = np.array([[positions_dict[i][0], positions_dict[i][1]] for i in range(len(node_names))])
print(f"Layout computed using Graph.Reshape with spring2d layout")

## Find Bridges (Cut Edges)

topologic_fast provides `Graph.Bridges` which uses Tarjan's algorithm to find all bridges. A bridge is an edge whose removal disconnects the graph.

In [ ]:
# Find bridges using topologic_fast's native implementation (Tarjan's algorithm)
bridge_edges = graph.Bridges()

# Convert bridge edges to (i, j) tuples for visualization
bridges = []
for edge in bridge_edges:
    verts = edge.Vertices()
    if len(verts) == 2:
        coords1 = verts[0].Coordinates()
        coords2 = verts[1].Coordinates()
        # Find vertex indices by matching positions
        i = min(range(len(positions)), key=lambda x: abs(positions[x][0] - coords1[0]) + abs(positions[x][1] - coords1[1]))
        j = min(range(len(positions)), key=lambda x: abs(positions[x][0] - coords2[0]) + abs(positions[x][1] - coords2[1]))
        bridges.append((min(i, j), max(i, j)))

print(f"Found {len(bridges)} bridge(s) in the graph (computed with topologic_fast):")
print("=" * 60)
for idx, (u, v) in enumerate(bridges, 1):
    print(f"{idx}. {node_names[u]} <-> {node_names[v]}")

## Find Cut Vertices (Articulation Points)

topologic_fast provides `Graph.CutVertices` which uses a modified Tarjan's algorithm. A cut vertex is a vertex whose removal disconnects the graph.

In [ ]:
# Find cut vertices using topologic_fast's native implementation (Tarjan's algorithm)
cut_vertex_objs = graph.CutVertices()

# Convert cut vertices to indices for visualization
graph_vertices = graph.Vertices()
cut_vertices = []
for cv in cut_vertex_objs:
    cv_coords = cv.Coordinates()
    # Find vertex index by matching positions
    idx = min(range(len(positions)), key=lambda x: abs(positions[x][0] - cv_coords[0]) + abs(positions[x][1] - cv_coords[1]))
    cut_vertices.append(idx)

cut_vertices = sorted(cut_vertices)

print(f"Found {len(cut_vertices)} cut vertex/vertices in the graph (computed with topologic_fast):")
print("=" * 60)
for i, v in enumerate(cut_vertices, 1):
    # Count number of neighbors
    neighbors = sum(adjacencyMatrix[v])
    print(f"{i}. {node_names[v]} ({neighbors} connections)")

## Visualize Bridges

Bridges are highlighted in red with thicker lines.

In [ ]:
def visualize_bridges(positions, adj_matrix, node_names, bridges):
    """
    Visualize the graph with bridges highlighted.
    """
    fig = go.Figure()
    bridge_set = set(bridges)
    
    # Draw non-bridge edges first
    for i in range(len(adj_matrix)):
        for j in range(i+1, len(adj_matrix)):
            if adj_matrix[i][j] == 1:
                edge_key = (min(i, j), max(i, j))
                is_bridge = edge_key in bridge_set
                
                if not is_bridge:
                    fig.add_trace(go.Scatter(
                        x=[positions[i, 0], positions[j, 0]],
                        y=[positions[i, 1], positions[j, 1]],
                        mode='lines',
                        line=dict(color='lightgray', width=1),
                        hoverinfo='skip',
                        showlegend=False
                    ))
    
    # Draw bridge edges
    for (i, j) in bridges:
        fig.add_trace(go.Scatter(
            x=[positions[i, 0], positions[j, 0]],
            y=[positions[i, 1], positions[j, 1]],
            mode='lines',
            line=dict(color='red', width=4),
            hoverinfo='text',
            hovertext=f'BRIDGE: {node_names[i]} <-> {node_names[j]}',
            showlegend=False,
            name='Bridge'
        ))
    
    # Draw vertices
    x_coords = [positions[i, 0] for i in range(len(node_names))]
    y_coords = [positions[i, 1] for i in range(len(node_names))]
    
    fig.add_trace(go.Scatter(
        x=x_coords,
        y=y_coords,
        mode='markers+text',
        marker=dict(
            size=16,
            color='lightblue',
            line=dict(color='darkblue', width=2)
        ),
        text=node_names,
        textposition='top center',
        textfont=dict(size=8),
        hovertext=node_names,
        hoverinfo='text',
        showlegend=False
    ))
    
    fig.update_layout(
        title=f'Graph Bridges (Cut Edges) - {len(bridges)} bridge(s) found<br><sub>Red edges are bridges - removing them disconnects the graph</sub>',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=1000,
        height=800,
        showlegend=False,
        hovermode='closest',
        plot_bgcolor='white'
    )
    
    return fig

fig = visualize_bridges(positions, adjacencyMatrix, node_names, bridges)
fig.show()

## Visualize Cut Vertices

Cut vertices are highlighted in red with larger markers.

In [ ]:
def visualize_cut_vertices(positions, adj_matrix, node_names, cut_vertices):
    """
    Visualize the graph with cut vertices highlighted.
    """
    fig = go.Figure()
    cut_set = set(cut_vertices)
    
    # Draw edges
    for i in range(len(adj_matrix)):
        for j in range(i+1, len(adj_matrix)):
            if adj_matrix[i][j] == 1:
                fig.add_trace(go.Scatter(
                    x=[positions[i, 0], positions[j, 0]],
                    y=[positions[i, 1], positions[j, 1]],
                    mode='lines',
                    line=dict(color='lightgray', width=2),
                    hoverinfo='skip',
                    showlegend=False
                ))
    
    # Draw non-cut vertices
    non_cut_indices = [i for i in range(len(node_names)) if i not in cut_set]
    non_cut_x = [positions[i, 0] for i in non_cut_indices]
    non_cut_y = [positions[i, 1] for i in non_cut_indices]
    non_cut_names = [node_names[i] for i in non_cut_indices]
    
    fig.add_trace(go.Scatter(
        x=non_cut_x,
        y=non_cut_y,
        mode='markers+text',
        marker=dict(
            size=10,
            color='lightgray',
            line=dict(color='gray', width=1)
        ),
        text=non_cut_names,
        textposition='top center',
        textfont=dict(size=8),
        hovertext=non_cut_names,
        hoverinfo='text',
        showlegend=False,
        name='Regular vertex'
    ))
    
    # Draw cut vertices
    cut_x = [positions[i, 0] for i in cut_vertices]
    cut_y = [positions[i, 1] for i in cut_vertices]
    cut_names = [node_names[i] for i in cut_vertices]
    cut_labels = [f"{node_names[i]} (CUT VERTEX)" for i in cut_vertices]
    
    fig.add_trace(go.Scatter(
        x=cut_x,
        y=cut_y,
        mode='markers+text',
        marker=dict(
            size=20,
            color='red',
            symbol='diamond',
            line=dict(color='darkred', width=2)
        ),
        text=cut_names,
        textposition='top center',
        textfont=dict(size=9, color='red'),
        hovertext=cut_labels,
        hoverinfo='text',
        showlegend=False,
        name='Cut vertex'
    ))
    
    fig.update_layout(
        title=f'Cut Vertices (Articulation Points) - {len(cut_vertices)} found<br><sub>Red diamonds are cut vertices - removing them disconnects the graph</sub>',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=1000,
        height=800,
        showlegend=False,
        hovermode='closest',
        plot_bgcolor='white'
    )
    
    return fig

fig = visualize_cut_vertices(positions, adjacencyMatrix, node_names, cut_vertices)
fig.show()

## Combined Visualization

Show both bridges and cut vertices together.

In [ ]:
def visualize_bridges_and_cuts(positions, adj_matrix, node_names, bridges, cut_vertices):
    """
    Visualize both bridges and cut vertices together.
    """
    fig = go.Figure()
    bridge_set = set(bridges)
    cut_set = set(cut_vertices)
    
    # Draw non-bridge edges first
    for i in range(len(adj_matrix)):
        for j in range(i+1, len(adj_matrix)):
            if adj_matrix[i][j] == 1:
                edge_key = (min(i, j), max(i, j))
                is_bridge = edge_key in bridge_set
                
                if not is_bridge:
                    fig.add_trace(go.Scatter(
                        x=[positions[i, 0], positions[j, 0]],
                        y=[positions[i, 1], positions[j, 1]],
                        mode='lines',
                        line=dict(color='lightgray', width=1),
                        hoverinfo='skip',
                        showlegend=False
                    ))
    
    # Draw bridge edges
    for (i, j) in bridges:
        fig.add_trace(go.Scatter(
            x=[positions[i, 0], positions[j, 0]],
            y=[positions[i, 1], positions[j, 1]],
            mode='lines',
            line=dict(color='orange', width=5),
            hoverinfo='text',
            hovertext=f'BRIDGE: {node_names[i]} <-> {node_names[j]}',
            showlegend=False
        ))
    
    # Draw non-cut vertices
    non_cut_indices = [i for i in range(len(node_names)) if i not in cut_set]
    non_cut_x = [positions[i, 0] for i in non_cut_indices]
    non_cut_y = [positions[i, 1] for i in non_cut_indices]
    non_cut_names = [node_names[i] for i in non_cut_indices]
    
    fig.add_trace(go.Scatter(
        x=non_cut_x,
        y=non_cut_y,
        mode='markers+text',
        marker=dict(
            size=14,
            color='lightblue',
            line=dict(color='darkblue', width=1)
        ),
        text=non_cut_names,
        textposition='top center',
        textfont=dict(size=7),
        hovertext=non_cut_names,
        hoverinfo='text',
        showlegend=False
    ))
    
    # Draw cut vertices
    if cut_vertices:
        cut_x = [positions[i, 0] for i in cut_vertices]
        cut_y = [positions[i, 1] for i in cut_vertices]
        cut_names = [node_names[i] for i in cut_vertices]
        
        fig.add_trace(go.Scatter(
            x=cut_x,
            y=cut_y,
            mode='markers+text',
            marker=dict(
                size=22,
                color='red',
                symbol='diamond',
                line=dict(color='darkred', width=2)
            ),
            text=cut_names,
            textposition='top center',
            textfont=dict(size=9, color='darkred'),
            hovertext=[f"{name} (CUT VERTEX)" for name in cut_names],
            hoverinfo='text',
            showlegend=False
        ))
    
    # Add legend manually
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=14, color='red', symbol='diamond'),
        name='Cut Vertex',
        showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='lines',
        line=dict(color='orange', width=5),
        name='Bridge',
        showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=14, color='lightblue'),
        name='Regular Vertex',
        showlegend=True
    ))
    
    fig.update_layout(
        title=f'Bridges and Cut Vertices<br><sub>{len(bridges)} bridge(s), {len(cut_vertices)} cut vertex/vertices</sub>',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=1000,
        height=800,
        legend=dict(x=1.02, y=1),
        hovermode='closest',
        plot_bgcolor='white'
    )
    
    return fig

fig = visualize_bridges_and_cuts(positions, adjacencyMatrix, node_names, bridges, cut_vertices)
fig.show()

## Analysis of Critical Points

In [ ]:
print("ANALYSIS OF CRITICAL POINTS IN THE GRAPH")
print("=" * 60)

print(f"\nTotal vertices: {len(node_names)}")
print(f"Total edges: {sum(sum(row) for row in adjacencyMatrix) // 2}")
print(f"Graph diameter: {graph.Diameter()}")
print(f"Graph density: {graph.Density():.4f}")

print(f"\n--- BRIDGES (Cut Edges) ---")
print(f"Number of bridges: {len(bridges)}")
if bridges:
    print("\nBridges identify critical connections. Removing a bridge disconnects the graph.")
    for (i, j) in bridges:
        print(f"  - {node_names[i]} <-> {node_names[j]}")
else:
    print("No bridges found - the graph has redundant paths between all nodes.")

print(f"\n--- CUT VERTICES (Articulation Points) ---")
print(f"Number of cut vertices: {len(cut_vertices)}")
if cut_vertices:
    print("\nCut vertices are critical nodes. Removing one disconnects the graph.")
    for v in cut_vertices:
        degree = sum(adjacencyMatrix[v])
        print(f"  - {node_names[v]} (degree: {degree})")
else:
    print("No cut vertices found - the graph is 2-connected.")

# Network robustness assessment
print("\n--- NETWORK ROBUSTNESS ---")
n_vertices = len(node_names)
n_edges = sum(sum(row) for row in adjacencyMatrix) // 2
cut_ratio = len(cut_vertices) / n_vertices if n_vertices > 0 else 0
bridge_ratio = len(bridges) / n_edges if n_edges > 0 else 0

print(f"Cut vertex ratio: {cut_ratio:.2%} of vertices are critical")
print(f"Bridge ratio: {bridge_ratio:.2%} of edges are critical")

if cut_ratio < 0.1 and bridge_ratio < 0.1:
    print("\nAssessment: The network is ROBUST with good redundancy.")
elif cut_ratio < 0.25 or bridge_ratio < 0.25:
    print("\nAssessment: The network has MODERATE resilience.")
else:
    print("\nAssessment: The network is VULNERABLE - many critical points.")

## Using topologic_fast Graph Methods

Let's also demonstrate the available graph methods in topologic_fast.

In [ ]:
# Demonstrate available topologic_fast Graph methods
print("topologic_fast Graph Methods:")
print("=" * 60)

print(f"\nGraph.Order() - Number of vertices: {graph.Order()}")
print(f"Graph.Size() - Number of edges: {graph.Size()}")
print(f"Graph.Density() - Graph density: {graph.Density():.4f}")
print(f"Graph.Diameter() - Longest shortest path: {graph.Diameter()}")
print(f"Graph.IsBipartite() - Is bipartite: {graph.IsBipartite()}")
print(f"Graph.IsComplete() - Is complete graph: {graph.IsComplete()}")

# Get adjacency matrix
adj_mat = graph.AdjacencyMatrix()
print(f"\nGraph.AdjacencyMatrix() - {len(adj_mat)}x{len(adj_mat)} matrix")

# Test shortest path between two rooms
v_list = graph.Vertices()
v_start = v_list[0]  # First vertex
v_end = v_list[13]   # Lobby (index 13)

distance = graph.Distance(v_start, v_end)
print(f"\nDistance from {node_names[0]} to {node_names[13]}: {distance} steps")

# Vertex degrees for critical nodes
if cut_vertices:
    print("\nVertex degrees for cut vertices:")
    for cv in cut_vertices:
        degree = graph.VertexDegree(v_list[cv])
        print(f"  {node_names[cv]}: degree = {degree}")

## Summary

This notebook demonstrated:

1. **Creating graphs from adjacency matrices** using `tf.Graph.ByAdjacencyMatrix`
2. **Finding bridges (cut edges)** using `tf.Graph.Bridges` - Tarjan's algorithm
3. **Finding cut vertices (articulation points)** using `tf.Graph.CutVertices` - modified Tarjan's algorithm
4. **Using Graph.Reshape** for spring layout visualization
5. **Visualizing critical graph elements** with Plotly
6. **Assessing network robustness** based on critical point analysis

### Key Findings:
- Bridges are edges whose removal disconnects the graph
- Cut vertices are nodes whose removal disconnects the graph
- These critical points reveal vulnerabilities in the network

### topologic_fast Methods Used:
- `Graph.ByAdjacencyMatrix(matrix)` - Creates graph from adjacency matrix
- `Graph.Bridges()` - Finds all bridge edges using Tarjan's algorithm
- `Graph.CutVertices()` - Finds all articulation points using Tarjan's algorithm
- `Graph.Reshape(layout_type, iterations)` - Force-directed spring layout

### Applications:
- Network reliability analysis
- Infrastructure vulnerability assessment
- Building evacuation planning
- Transportation network design